# 3주차 실습 (2) - PCA from scratch

NumPy만으로 PCA를 구현한다: **중심화 → 공분산 → 고유분해 → 투영**.
합성 데이터로 설명분산비를 확인하고, (설치돼 있으면) sklearn과 대조한다.

## Step 1. PCA 클래스 구현

공분산 행렬은 대칭이므로 `np.linalg.eigh`를 쓴다 (실수 고유값 보장).
고유값이 큰 방향 = 분산이 큰 방향 = 우선 남길 주성분.

In [ ]:
import numpy as np


class PCA:
    def __init__(self, n_components):
        self.n_components = n_components
        self.mean = None
        self.components = None          # (k, d)
        self.eigenvalues = None
        self.explained_variance_ratio_ = None

    def fit(self, X):
        self.mean = np.mean(X, axis=0)          # 1. 중심화
        Xc = X - self.mean
        cov = np.cov(Xc, rowvar=False)          # 2. 공분산
        eigenvalues, eigenvectors = np.linalg.eigh(cov)  # 3. 고유분해
        order = np.argsort(eigenvalues)[::-1]   # 4. 내림차순 정렬
        eigenvalues = eigenvalues[order]
        eigenvectors = eigenvectors[:, order]
        self.components = eigenvectors[:, : self.n_components].T  # 5. 상위 k
        self.eigenvalues = eigenvalues[: self.n_components]
        self.explained_variance_ratio_ = self.eigenvalues / np.sum(eigenvalues)
        return self

    def transform(self, X):
        return (X - self.mean) @ self.components.T

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, X_reduced):
        """저차원 표현에서 원 공간으로 복원 (근사)."""
        return X_reduced @ self.components + self.mean

## Step 2. 합성 데이터로 테스트

3D 이지만 실질적으로 원(2D 매니폴드) 위에 놓인 데이터를 만든다.
`x3`은 `x1, x2`의 선형 결합이라 잉여 차원 → PCA가 2성분으로 대부분의 분산을 잡아야 한다.

In [ ]:
rng = np.random.default_rng(42)
n = 500

t = rng.uniform(0, 2 * np.pi, n)
x1 = 3 * np.cos(t) + rng.normal(0, 0.2, n)
x2 = 3 * np.sin(t) + rng.normal(0, 0.2, n)
x3 = 0.5 * x1 + 0.3 * x2 + rng.normal(0, 0.1, n)   # 잉여 차원
X = np.column_stack([x1, x2, x3])

pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X)

print(f"원본 shape:  {X.shape}")
print(f"축소 shape:  {X_reduced.shape}")
print(f"설명분산비:  {np.round(pca.explained_variance_ratio_, 4)}")
print(f"누적 분산:   {pca.explained_variance_ratio_.sum():.4f}")

## Step 3. 복원 오차 (Reconstruction Error)

복원 오차 = 버린 성분에 담긴 분산의 합. 저차원으로 눌렀다 되돌렸을 때 잃은 정보의 크기.

In [ ]:
def reconstruction_mse(X, pca):
    X_hat = pca.inverse_transform(pca.transform(X))
    return np.mean((X - X_hat) ** 2)


print(f"복원 MSE (2성분): {reconstruction_mse(X, pca):.6f}")

## Step 4. sklearn 과 대조

직접 구현한 설명분산비가 라이브러리와 일치하는지 확인한다 (미설치면 건너뜀).

In [ ]:
try:
    from sklearn.decomposition import PCA as SKPCA

    sk = SKPCA(n_components=2).fit(X)
    print(f"우리 설명분산비:    {np.round(pca.explained_variance_ratio_, 6)}")
    print(f"sklearn 설명분산비: {np.round(sk.explained_variance_ratio_, 6)}")
except ImportError:
    print("sklearn 미설치 - pip install scikit-learn 하면 대조 가능")

## (선택) Step 5. 시각화

2D로 투영한 결과를 산점도로 그려본다. matplotlib 이 있으면 실행된다.

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].scatter(X[:, 0], X[:, 1], s=8, alpha=0.6)
    ax[0].set_title("원본 (x1, x2)")
    ax[0].set_aspect("equal")
    ax[1].scatter(X_reduced[:, 0], X_reduced[:, 1], s=8, alpha=0.6, c="tab:orange")
    ax[1].set_title("PCA 2D 투영 (PC1, PC2)")
    ax[1].set_aspect("equal")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib 미설치 - pip install matplotlib 하면 그림 가능")

## 정리

- PCA = 좌표계를 돌려 분산이 큰 축(PC1, PC2...)에 정렬한 뒤 작은 축을 버리는 것.
- 핵심은 **공분산 행렬의 고유값·고유벡터** — 1주차 선형대수와 직결.
- 시각화(t-SNE/UMAP)와 달리 PCA는 전처리·압축에 강하고 수백만 샘플에도 적용 가능.
- 관련 개념 노트: 차원 축소 (PCA·t-SNE·UMAP) (Obsidian LLM_Wiki)